# HIV Epidemiological Transitions and Treatment Disparities in Southeast Asia (1990–2025)
### A Comparative Longitudinal Analysis of Myanmar Against Regional and Global Benchmarks

**Investigator:** Dr. Aung Chit Moe — Medical Doctor (MBBS, Myanmar) & Healthcare Data Analyst  
**Data Sources:** UNAIDS 2026 Epidemiological Estimates, WHO Global Health Observatory, World Bank WDI  
**Target:** Reproducible Clinical Research & Epidemiological Analytics Portfolio  

---

## 1. Executive Summary & Clinical Background
Over the past three decades, antiretroviral therapy (ART) has fundamentally changed HIV from an acute fatal illness into a manageable chronic disease. However, public health progress has been remarkably uneven across Southeast Asia.

This study investigates:
1. **Long-Term Trajectories (1990–2025):** How HIV incidence, adult prevalence, and AIDS mortality evolved across 10 Southeast Asian nations and global benchmarks.
2. **Myanmar Specifics:** The rapid 2010–2020 ART expansion (12% to 69%), the 2021–2022 disruption (+11.6% mortality rebound), and the prevalence paradox.
3. **Treatment Disparities & Cascade:** UNAIDS 95-95-95 performance in 2025 and regional archetype classification (Accelerated Control vs. Counter-Epidemics).
4. **Econometric Lagged Association:** Country fixed-effects panel modeling of ART expansion on AIDS mortality reductions.
5. **Socioeconomic Determinants:** The decoupling of GDP per capita from treatment coverage in donor-supported health systems.

In [1]:
# Setup and library imports
import os
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Visual configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

DB_PATH = os.path.abspath(os.path.join('..', 'data', 'processed', 'hiv_surveillance.db'))
print(f"Database verified: {os.path.basename(DB_PATH)} ({os.path.getsize(DB_PATH):,} bytes)")

Database verified: hiv_surveillance.db (1,200,128 bytes)


## 2. Structured Data Ingestion via SQLite
We query our harmonized surveillance database (`hiv_surveillance.db`), loading the master analytical view `v_southeast_asia_surveillance`.

In [2]:
conn = sqlite3.connect(DB_PATH)

# Query table verification counts
tables = ['dim_country', 'fact_hiv_estimates', 'fact_treatment_cascade', 'fact_socioeconomic']
for tbl in tables:
    cnt = pd.read_sql_query(f"SELECT COUNT(*) as n FROM {tbl}", conn)['n'].iloc[0]
    print(f"Table {tbl:<25}: {cnt:,} rows")

# Load full Southeast Asia & Benchmarks dataset
df_surv = pd.read_sql_query("SELECT * FROM v_southeast_asia_surveillance ORDER BY country_name, year", conn)
conn.close()

print(f"\nMaster Analytical View loaded: {df_surv.shape[0]} rows across {df_surv['iso3'].nunique()} entities.")
df_surv.head(3)

Table dim_country              : 184 rows
Table fact_hiv_estimates       : 6,624 rows
Table fact_treatment_cascade   : 2,944 rows
Table fact_socioeconomic       : 6,624 rows

Master Analytical View loaded: 684 rows across 19 entities.


,iso3,country_name,region,income_group,analysis_role,is_southeast_asia,is_benchmark,year,plhiv,adult_prevalence_pct,aids_deaths,new_infections,incidence_rate_per1000,art_coverage_pct,treatment_gap_pct,unreached_plhiv,diagnosed_1st95_pct,on_art_2nd95_pct,virally_suppressed_3rd95_pct,gdp_per_capita_usd,gdp_per_capita_ppp,population_total,mortality_rate_in_plhiv_pct,aids_mortality_rate_per100k_pop,incidence_to_mortality_ratio
0,UNAAP,Asia and the Pacific,Regional Aggregate,Aggregate,Regional Benchmark,0,1,1990,940000.000,0.100,16000.000,440000.000,0.150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.702,NaN,27.500
1,UNAAP,Asia and the Pacific,Regional Aggregate,Aggregate,Regional Benchmark,0,1,1991,1500000.000,0.100,29000.000,600000.000,0.200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.933,NaN,20.690
2,UNAAP,Asia and the Pacific,Regional Aggregate,Aggregate,Regional Benchmark,0,1,1992,2200000.000,0.100,48000.000,720000.000,0.240,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.182,NaN,15.000


## 3. SQL Query 1: Myanmar 36-Year Trajectory (1990–2025)
We inspect Myanmar's empirical progression across the pre-treatment epidemic wave, ART scale-up, and the political/COVID disruption.

In [3]:
conn = sqlite3.connect(DB_PATH)
query_mmr = """
SELECT 
    year,
    adult_prevalence_pct AS prev_pct,
    plhiv,
    new_infections AS new_inf,
    aids_deaths AS deaths,
    art_coverage_pct AS art_cov,
    treatment_gap_pct AS gap_pct,
    ROUND(mortality_rate_in_plhiv_pct, 2) AS mort_rate_plhiv,
    ROUND(gdp_per_capita_usd, 0) AS gdp_pc
FROM v_southeast_asia_surveillance
WHERE iso3 = 'MMR'
ORDER BY year DESC;
"""
df_mmr = pd.read_sql_query(query_mmr, conn)
conn.close()

# Display recent 16 years (2010-2025)
df_mmr[df_mmr['year'] >= 2010]

,year,prev_pct,plhiv,new_inf,deaths,art_cov,gap_pct,mort_rate_plhiv,gdp_pc
0,2025,0.900,310000.000,9500.000,4400.000,73.000,27.000,1.420,1489.000
1,2024,0.900,300000.000,9900.000,4400.000,72.000,28.000,1.470,1359.000
2,2023,0.900,300000.000,10000.000,4600.000,72.000,28.000,1.530,1233.000
3,2022,0.900,300000.000,11000.000,4800.000,70.000,30.000,1.600,1158.000
4,2021,0.900,290000.000,11000.000,4300.000,66.000,34.000,1.480,1243.000
5,2020,0.900,290000.000,10000.000,4400.000,69.000,31.000,1.520,1490.000
6,2019,0.900,280000.000,11000.000,4800.000,65.000,35.000,1.710,1426.000
7,2018,0.900,280000.000,12000.000,5400.000,60.000,40.000,1.930,1298.000
8,2017,0.900,270000.000,13000.000,6100.000,53.000,47.000,2.260,1273.000
9,2016,0.900,270000.000,14000.000,6900.000,47.000,53.000,2.560,1229.000


### Clinical Epidemiological Interpretation: Myanmar Dynamics
1. **Prevalence Paradox:** Adult HIV prevalence remained flat at `0.9%` while total PLHIV expanded from `250,000` to `310,000`. As clinicians understand, effective ART extends survival, causing living prevalence to expand even as new transmissions contract.
2. **The 2021–2022 Disruption:** Between 2010 and 2020, ART expanded from `12%` to `69%`, causing deaths to drop from `12,000` to `4,400`. In 2021, amidst political turmoil and COVID-19 strain, ART coverage dropped to `66%`, resulting in an immediate mortality rebound to `4,800` deaths in 2022 (+11.6%). Recovery followed, reaching `73%` coverage by 2025.

## 4. SQL Query 2: Regional Fast-Track Progress (2010 vs 2025)
Evaluating relative changes in AIDS deaths and new infections alongside ART coverage gains across Southeast Asia.

In [4]:
conn = sqlite3.connect(DB_PATH)
query_fasttrack = """
WITH b AS (
    SELECT iso3, country_name, aids_deaths AS d_10, new_infections AS i_10, art_coverage_pct AS art_10
    FROM v_southeast_asia_surveillance WHERE year = 2010
),
c AS (
    SELECT iso3, aids_deaths AS d_25, new_infections AS i_25, art_coverage_pct AS art_25
    FROM v_southeast_asia_surveillance WHERE year = 2025
)
SELECT 
    b.country_name,
    b.d_10 AS deaths_2010,
    c.d_25 AS deaths_2025,
    ROUND(((c.d_25 - b.d_10) * 100.0 / b.d_10), 1) AS pct_change_deaths,
    b.i_10 AS inf_2010,
    c.i_25 AS inf_2025,
    ROUND(((c.i_25 - b.i_10) * 100.0 / b.i_10), 1) AS pct_change_infections,
    b.art_10 AS art_2010,
    c.art_25 AS art_2025,
    ROUND(c.art_25 - b.art_10, 1) AS art_gain_pp
FROM b JOIN c ON b.iso3 = c.iso3
WHERE b.d_10 IS NOT NULL
ORDER BY pct_change_deaths ASC;
"""
df_ft = pd.read_sql_query(query_fasttrack, conn)
conn.close()
df_ft

,country_name,deaths_2010,deaths_2025,pct_change_deaths,inf_2010,inf_2025,pct_change_infections,art_2010,art_2025,art_gain_pp
0,Thailand,41000.000,8400.000,-79.500,21000.000,7300.000,-65.200,27.000,82.000,55.000
1,Caribbean,14000.000,4800.000,-65.700,19000.000,13000.000,-31.600,18.000,73.000,55.000
2,Myanmar,12000.000,4400.000,-63.300,17000.000,9500.000,-44.100,12.000,73.000,61.000
3,Eastern and southern Africa,630000.000,240000.000,-61.900,1100000.000,420000.000,-61.800,25.000,85.000,60.000
4,Western and central Africa,280000.000,110000.000,-60.700,340000.000,190000.000,-44.100,17.000,74.000,57.000
5,Asia and the Pacific,300000.000,120000.000,-60.000,350000.000,280000.000,-20.000,15.000,72.000,57.000
6,Global,1300000.000,570000.000,-56.200,2100000.000,1200000.000,-42.900,24.000,78.000,54.000
7,Cambodia,1700.000,1000.000,-41.200,2300.000,960.000,-58.300,50.000,95.000,45.000
8,Malaysia,3400.000,2000.000,-41.200,6100.000,2900.000,-52.500,18.000,63.000,45.000
9,Latin America,46000.000,28000.000,-39.100,110000.000,140000.000,27.300,34.000,71.000,37.000


## 5. Visualizing the Epidemiological Evidence Base
We inspect and verify the 6 publication-grade figures (300 DPI) generated by our pipeline in `figures/`.

In [5]:
from PIL import Image as PILImage

fig_files = [
    '../figures/fig1_myanmar_longitudinal_trajectory.png',
    '../figures/fig2_southeast_asia_mortality_infection_divergence.png',
    '../figures/fig3_unaids_95_95_95_cascade_2025.png',
    '../figures/fig4_lagged_art_mortality_dose_response.png',
    '../figures/fig5_gdp_vs_art_coverage_disparity.png',
    '../figures/fig6_southeast_asia_art_trajectories_2010_2025.png'
]

fig_meta = []
for fig_path in fig_files:
    with PILImage.open(fig_path) as im:
        dpi = im.info.get('dpi', (0, 0))
        fig_meta.append({
            'figure_file': os.path.basename(fig_path),
            'dimensions_px': f'{im.width}x{im.height}',
            'dpi': round(dpi[0]),
            'size_kb': round(os.path.getsize(fig_path) / 1024, 1)
        })
pd.DataFrame(fig_meta)

,figure_file,dimensions_px,dpi,size_kb
0,fig1_myanmar_longitudinal_trajectory.png,3415x2748,300,724.500
1,fig2_southeast_asia_mortality_infection_divergence.png,4482x2154,300,353.800
2,fig3_unaids_95_95_95_cascade_2025.png,4622x2083,300,408.400
3,fig4_lagged_art_mortality_dose_response.png,4618x2140,300,895.100
4,fig5_gdp_vs_art_coverage_disparity.png,3070x2129,300,396.400
5,fig6_southeast_asia_art_trajectories_2010_2025.png,3725x2131,300,676.400


## 6. Econometric Panel Regressions: Fixed Effects Estimation
We estimate country fixed-effects regressions to test the lagged protective association of ART on crude mortality rate in PLHIV.

In [6]:
# Load processed SEA panel (2010-2025)
sea_panel = pd.read_csv('../data/processed/southeast_asia_hiv_panel_2010_2025.csv')
reg_countries = ['MMR', 'THA', 'VNM', 'KHM', 'IDN', 'MYS', 'PHL', 'LAO', 'TLS']

pdata = sea_panel[sea_panel['iso3'].isin(reg_countries)].dropna(subset=['mortality_rate_in_plhiv_pct', 'art_coverage_pct']).copy()
pdata['log_gdp'] = np.log(pdata['gdp_per_capita_ppp'])

# Within-country mean demeaning for Model 2 (1-Year Lag)
pdata_fe = pdata.dropna(subset=['art_coverage_lag1']).copy()
y = pdata_fe['mortality_rate_in_plhiv_pct'] - pdata_fe.groupby('iso3')['mortality_rate_in_plhiv_pct'].transform('mean')
X = pdata_fe['art_coverage_lag1'] - pdata_fe.groupby('iso3')['art_coverage_lag1'].transform('mean')
X = sm.add_constant(X)

fe_model2 = sm.OLS(y, X).fit(cov_type='HC1')
print(fe_model2.summary())

                                 OLS Regression Results                                
Dep. Variable:     mortality_rate_in_plhiv_pct   R-squared:                       0.687
Model:                                     OLS   Adj. R-squared:                  0.685
Method:                          Least Squares   F-statistic:                     201.7
Date:                         Sun, 11 Oct 2026   Prob (F-statistic):           1.96e-28
Time:                                 17:27:35   Log-Likelihood:                -100.66
No. Observations:                          135   AIC:                             205.3
Df Residuals:                              133   BIC:                             211.1
Df Model:                                    1                                         
Covariance Type:                           HC1                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------

## 7. Conclusions & Research Discussion
1. **Primary Finding:** Across Southeast Asia, each **10 percentage point expansion in national ART coverage** reduces the mortality rate among people living with HIV by **~0.49 percentage points** in the subsequent year ($t = -14.20, p < 0.00001, R^2_{\text{within}} = 0.687$).
2. **Myanmar Resilience & Vulnerability:** Myanmar demonstrates that rapid vertical ART decentralization can drive dramatic mortality reductions in low-income settings. However, clinical supply stability is essential; political and health system disruptions immediately manifest as excess mortality rebounds.
3. **The Counter-Epidemic in the Philippines:** With a **+639% surge in new infections** and **+440% increase in AIDS deaths**, the Philippines requires urgent structural interventions and testing expansion.
4. **Policy Decoupling:** In concentrated epidemics, national GDP does not dictate HIV success. Lower-income countries with strong vertical programs (Cambodia, Myanmar) outperform decentralized middle-income countries (Indonesia, Philippines).